# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Random Forest Classifier

**Why:**

- It combines predictions from multiple decision trees.
- It can learn relationships between multiple features.
- Week 4 showed that CTR should be interpreted relative to search position.
- Therefore, Random Forest is a reasonable method for learning these relationships and producing a ranking for refresh review.
- Its predicted probability can be used as the opportunity score.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
# Checking the time span of the warehouse data
date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

date_range

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,number_of_days
0,2025-01-27,2026-06-30,520


In [6]:
# Checking the monthly coverage
monthly_counts = con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

monthly_counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows,clients,content_pages
0,2025-01-01,1297,2,476
1,2025-02-01,75985,3,5903
2,2025-03-01,167859,4,10374
3,2025-04-01,285114,4,13046
4,2025-05-01,349923,4,14887
5,2025-06-01,329201,9,16399
6,2025-07-01,469794,16,27945
7,2025-08-01,704962,15,37204
8,2025-09-01,845813,23,53127
9,2025-10-01,2165471,31,110339


In [7]:
# Count how many unique content pages appear in both March and April 2026.
# This tells us whether March pages can realistically be followed into the
# next month for an out-of-time evaluation.

march_april_overlap = con.sql(f"""
    WITH march_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
    ),

    april_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date >= DATE '2026-04-01'
          AND report_date < DATE '2026-05-01'
    )

    SELECT
        COUNT(*) AS march_pages,
        COUNT(april.content_hash_id) AS pages_also_in_april,
        ROUND(
            100.0 * COUNT(april.content_hash_id) / COUNT(*),
            2
        ) AS pct_march_pages_also_in_april
    FROM march_pages march
    LEFT JOIN april_pages april
        ON march.client_hash_id = april.client_hash_id
       AND march.content_hash_id = april.content_hash_id
""").df()

march_april_overlap

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_also_in_april,pct_march_pages_also_in_april
0,331437,331436,100.0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.